In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import time
import os
import urllib.request
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import pandas as pd

# 웹드라이버 설정
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument('--headless')  # 브라우저 창을 띄우지 않음
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 설치 및 초기화
service = ChromeService(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.implicitly_wait(3)

# Naver API key 입력
client_id = '2KJ9WMM3SekPwby7w1MZ'
client_secret = 'YOUR_CREDENTIAL_HERE'

# 검색어 입력
keyword = input("검색할 키워드를 입력해주세요: ")
encText = urllib.parse.quote(keyword)

# 검색을 끝낼 페이지 입력
end = input("\n크롤링을 끝낼 위치를 입력해주세요. (기본값: 1, 최대값: 100): ")
end = int(end) if end else 1
print("\n 1 ~", end, "페이지까지 크롤링을 진행합니다")

# 한번에 가져올 페이지 입력
display = input("\n한번에 가져올 페이지 개수를 입력해주세요. (기본값: 10, 최대값: 100): ")
display = int(display) if display else 10
print("\n한번에 가져올 페이지:", display, "페이지")

naver_urls = []
postdate = []
titles = []

# 네이버 블로그 검색
for start in range(1, end * display, display):
    url = f"https://openapi.naver.com/v1/search/blog?query={encText}&start={start}&display={display}"
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    
    if rescode == 200:
        response_body = response.read()
        data = json.loads(response_body.decode('utf-8'))['items']
        for row in data:
            if 'blog.naver' in row['link']:
                naver_urls.append(row['link'])
                postdate.append(row['postdate'])
                title = re.sub('<[^>]*>', '', row['title'])  # html 태그 제거
                titles.append(title)
        time.sleep(2)
    else:
        print("Error Code:", rescode)

# 블로그 본문 가져오기
contents = []

for url in naver_urls:
    try:
        print("Processing:", url)
        driver.get(url)
        time.sleep(5)  # 대기 시간 조정 가능

        # iframe 내부로 이동
        iframe = driver.find_element(By.ID, "mainFrame")
        driver.switch_to.frame(iframe)

        # 페이지 소스 가져오기
        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")

        # 본문 추출
        content_elements = html.select("div.se-main-container")
        content = ''.join([elem.get_text(separator=" ", strip=True) for elem in content_elements])

        # html 태그 제거
        content = re.sub('<[^>]*>', '', content)
        contents.append(content)

    except Exception as e:
        print("Error processing URL:", url)
        print(e)
        contents.append('error')

# DataFrame 생성 및 CSV 파일로 저장
news_df = pd.DataFrame({'title': titles, 'content': contents, 'date': postdate})
news_df.to_csv('blog.csv', index=False, encoding='utf-8-sig')

# 드라이버 종료
driver.quit()

print("크롤링이 완료되었습니다. 'blog.csv' 파일을 확인하세요.")


In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import time
import os
import urllib.request
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import pandas as pd

# 웹드라이버 설정
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument('--headless')  # 브라우저 창을 띄우지 않음
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 설치 및 초기화
service = ChromeService(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.implicitly_wait(3)

# Naver API key 입력
client_id = '2KJ9WMM3SekPwby7w1MZ'
client_secret = 'YOUR_CREDENTIAL_HERE'

# 검색어 입력
keyword = input("검색할 키워드를 입력해주세요: ")
encText = urllib.parse.quote(keyword)

# 검색을 끝낼 페이지 입력
end = input("\n크롤링을 끝낼 위치를 입력해주세요. (기본값: 1, 최대값: 100): ")
end = int(end) if end else 1
print("\n 1 ~", end, "페이지까지 크롤링을 진행합니다")

# 한번에 가져올 페이지 입력
display = input("\n한번에 가져올 페이지 개수를 입력해주세요. (기본값: 10, 최대값: 100): ")
display = int(display) if display else 10
print("\n한번에 가져올 페이지:", display, "페이지")

naver_urls = []
postdate = []
titles = []
use_locations = []
return_locations = []

# 네이버 블로그 검색
for start in range(1, end * display, display):
    url = f"https://openapi.naver.com/v1/search/blog?query={encText}&start={start}&display={display}"
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    
    if rescode == 200:
        response_body = response.read()
        data = json.loads(response_body.decode('utf-8'))['items']
        for row in data:
            if 'blog.naver' in row['link']:
                naver_urls.append(row['link'])
                postdate.append(row['postdate'])
                title = re.sub('<[^>]*>', '', row['title'])  # html 태그 제거
                titles.append(title)
        time.sleep(2)
    else:
        print("Error Code:", rescode)

# 블로그 본문 가져오기
contents = []

for url in naver_urls:
    try:
        print("Processing:", url)
        driver.get(url)
        time.sleep(5)  # 대기 시간 조정 가능

        # iframe 내부로 이동
        iframe = driver.find_element(By.ID, "mainFrame")
        driver.switch_to.frame(iframe)

        # 페이지 소스 가져오기
        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")

        # 본문 추출
        content_elements = html.select("div.se-main-container")
        content = ''.join([elem.get_text(separator=" ", strip=True) for elem in content_elements])

        # html 태그 제거
        content = re.sub('<[^>]*>', '', content)
        contents.append(content)

        # 이용지점 및 반납지점 추출
        use_location = '정보 없음'
        return_location = '정보 없음'

        use_match = re.search(r'(이용지점|출발지점)\s*:\s*(\S+)', content)
        return_match = re.search(r'(반납지점|도착지점)\s*:\s*(\S+)', content)
        
        if use_match:
            use_location = use_match.group(2)
        if return_match:
            return_location = return_match.group(2)

        use_locations.append(use_location)
        return_locations.append(return_location)

    except Exception as e:
        print("Error processing URL:", url)
        print(e)
        contents.append('error')
        use_locations.append('error')
        return_locations.append('error')

# DataFrame 생성 및 CSV 파일로 저장
news_df = pd.DataFrame({'title': titles, 'content': contents, 'date': postdate, 'use_location': use_locations, 'return_location': return_locations})
news_df.to_csv('blog2.csv', index=False, encoding='utf-8-sig')

# 드라이버 종료
driver.quit()

print("크롤링이 완료되었습니다. 'blog.csv' 파일을 확인하세요.")


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import time
import os
import urllib.request
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# 쏘카 지점 이름 불러오기
location_df = pd.read_csv('location_name.csv')
location_names = location_df['name'].tolist()  # Assuming the column name is 'name'

# 웹드라이버 설정
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument('--headless')  # 브라우저 창을 띄우지 않음
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 설치 및 초기화
service = ChromeService(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.implicitly_wait(3)

# Naver API key 입력
client_id = '2KJ9WMM3SekPwby7w1MZ'
client_secret = 'YOUR_CREDENTIAL_HERE'

# 검색어 입력
keyword = input("검색할 키워드를 입력해주세요: ")
encText = urllib.parse.quote(keyword)

# 검색을 끝낼 페이지 입력
end = input("\n크롤링을 끝낼 위치를 입력해주세요. (기본값: 1, 최대값: 100): ")
end = int(end) if end else 1
print("\n 1 ~", end, "페이지까지 크롤링을 진행합니다")

# 한번에 가져올 페이지 입력
display = input("\n한번에 가져올 페이지 개수를 입력해주세요. (기본값: 10, 최대값: 100): ")
display = int(display) if display else 10
print("\n한번에 가져올 페이지:", display, "페이지")

naver_urls = []
postdate = []
titles = []
location_texts = []

# 네이버 블로그 검색
for start in range(1, end * display, display):
    url = f"https://openapi.naver.com/v1/search/blog?query={encText}&start={start}&display={display}"
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    
    if rescode == 200:
        response_body = response.read()
        data = json.loads(response_body.decode('utf-8'))['items']
        for row in data:
            if 'blog.naver' in row['link']:
                naver_urls.append(row['link'])
                postdate.append(row['postdate'])
                title = re.sub('<[^>]*>', '', row['title'])  # html 태그 제거
                titles.append(title)
        time.sleep(2)
    else:
        print("Error Code:", rescode)

# 블로그 본문 가져오기
contents = []

for url in naver_urls:
    try:
        print("Processing:", url)
        driver.get(url)
        time.sleep(5)  # 대기 시간 조정 가능

        # iframe 내부로 이동
        iframe = driver.find_element(By.ID, "mainFrame")
        driver.switch_to.frame(iframe)

        # 페이지 소스 가져오기
        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")

        # 본문 추출
        content_elements = html.select("div.se-main-container")
        content = ''.join([elem.get_text(separator=" ", strip=True) for elem in content_elements])

        # html 태그 제거
        content = re.sub('<[^>]*>', '', content)
        contents.append(content)

        # 쏘카 지점 이름 추출
        location_text = ''
        for location in location_names:
            if location in content:
                location_text += location + ' '

        location_texts.append(location_text.strip())

    except Exception as e:
        print("Error processing URL:", url)
        print(e)
        contents.append('error')
        location_texts.append('error')

# DataFrame 생성 및 CSV 파일로 저장
news_df = pd.DataFrame({'title': titles, 'content': contents, 'date': postdate, 'locations': location_texts})
news_df.to_csv('blog_with_locations3.csv', index=False, encoding='utf-8-sig')

# 드라이버 종료
driver.quit()

print("크롤링이 완료되었습니다. 'blog_with_locations.csv' 파일을 확인하세요.")

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import time
import os
import urllib.request
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from PIL import Image
import pytesseract
import io

# Tesseract 경로 설정 (필요한 경우)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# 쏘카 지점 이름 불러오기
location_df = pd.read_csv('location_name.csv')
location_names = location_df['name'].tolist()  # Assuming the column name is 'name'

# 웹드라이버 설정
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument('--headless')  # 브라우저 창을 띄우지 않음
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 설치 및 초기화
service = ChromeService(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.implicitly_wait(3)

# Naver API key 입력
client_id = '2KJ9WMM3SekPwby7w1MZ'
client_secret = 'YOUR_CREDENTIAL_HERE'

# 검색어 입력
keyword = input("검색할 키워드를 입력해주세요: ")
encText = urllib.parse.quote(keyword)

# 검색을 끝낼 페이지 입력
end = input("\n크롤링을 끝낼 위치를 입력해주세요. (기본값: 1, 최대값: 100): ")
end = int(end) if end else 1
print("\n 1 ~", end, "페이지까지 크롤링을 진행합니다")

# 한번에 가져올 페이지 입력
display = input("\n한번에 가져올 페이지 개수를 입력해주세요. (기본값: 10, 최대값: 100): ")
display = int(display) if display else 10
print("\n한번에 가져올 페이지:", display, "페이지")

naver_urls = []
postdate = []
titles = []
location_texts = []

# 네이버 블로그 검색
for start in range(1, end * display, display):
    url = f"https://openapi.naver.com/v1/search/blog?query={encText}&start={start}&display={display}"
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    
    if rescode == 200:
        response_body = response.read()
        data = json.loads(response_body.decode('utf-8'))['items']
        for row in data:
            if 'blog.naver' in row['link']:
                naver_urls.append(row['link'])
                postdate.append(row['postdate'])
                title = re.sub('<[^>]*>', '', row['title'])  # html 태그 제거
                titles.append(title)
        time.sleep(2)
    else:
        print("Error Code:", rescode)

# 블로그 본문 가져오기
contents = []

for url in naver_urls:
    try:
        print("Processing:", url)
        driver.get(url)
        time.sleep(5)  # 대기 시간 조정 가능

        # iframe 내부로 이동
        iframe = driver.find_element(By.ID, "mainFrame")
        driver.switch_to.frame(iframe)

        # 페이지 소스 가져오기
        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")

        # 본문 추출
        content_elements = html.select("div.se-main-container")
        content = ''.join([elem.get_text(separator=" ", strip=True) for elem in content_elements])

        # html 태그 제거
        content = re.sub('<[^>]*>', '', content)
        contents.append(content)

        # 이미지에서 텍스트 추출
        img_tags = html.select("img")  # 블로그 글에서 모든 이미지 태그 선택
        location_text = ''
        
        for img_tag in img_tags:
            img_url = img_tag['src']
            try:
                # 이미지 다운로드
                img_response = requests.get(img_url)
                img = Image.open(io.BytesIO(img_response.content))

                # OCR을 사용해 이미지에서 텍스트 추출
                text = pytesseract.image_to_string(img, lang='kor')  # 한국어 사용
                for location in location_names:
                    if location in text:
                        location_text += location + ' '
            except Exception as e:
                print(f"Error processing image: {img_url}")
                print(e)

        location_texts.append(location_text.strip())

    except Exception as e:
        print("Error processing URL:", url)
        print(e)
        contents.append('error')
        location_texts.append('error')

# DataFrame 생성 및 CSV 파일로 저장
news_df = pd.DataFrame({'title': titles, 'content': contents, 'date': postdate, 'locations': location_texts})
news_df.to_csv('blog_with_locations2.csv', index=False, encoding='utf-8-sig')

# 드라이버 종료
driver.quit()

print("크롤링이 완료되었습니다. 'blog_with_locations.csv' 파일을 확인하세요.")